Step 1 — Load the frozen provisional classification file

In [ ]:
from pathlib import Path
import pandas as pd

In [ ]:

BASE_DIR = Path("/content/drive/MyDrive/GenAI-task-affordance-labor-outcomes")
PROVISIONAL_FILE = (
    BASE_DIR
    / "02_Task_Data_and_Classification"
    / "Output"
    / "Occupational_Task_Classification_Provisional_v6.csv"
)

provisional = pd.read_csv(PROVISIONAL_FILE)

print(f"Tasks: {len(provisional):,}")
print(f"Unique task IDs: {provisional['task_id'].nunique():,}")
print(f"Missing labels: {provisional['provisional_label'].isna().sum()}")

Step 2 — Remove the 20 locked pilot tasks from the validation sampling frame.

In [ ]:
PILOT_FILE = (
    BASE_DIR
    / "02_Task_Data_and_Classification"
    / "Output"
    / "Occupational_Task_Classification_Pilot_v6.csv"
)

pilot = pd.read_csv(PILOT_FILE)

validation_frame = provisional[
    ~provisional["task_id"].isin(pilot["task_id"])
].copy()

print(f"Pilot tasks excluded: {len(pilot):,}")
print(f"Validation sampling frame: {len(validation_frame):,}")
print(f"Unique task IDs: {validation_frame['task_id'].nunique():,}")

Step 3 — Define validation strata from the Pass-1 provisional category

In [ ]:
validation_summary = (
    validation_frame["provisional_label"]
    .value_counts()
    .rename("population_n")
    .rename_axis("provisional_label")
    .reset_index()
)

display(validation_summary)

Step 4 — Lock the validation sample allocation

In [ ]:
# Step 4 — Calculate validation allocation from current sampling frame

VALIDATION_TOTAL = 120
AUTOMATION_TARGET = 20

validation_summary["validation_n"] = 0

# Include all Indeterminate cases
indeterminate_mask = (
    validation_summary["provisional_label"] == "Indeterminate mechanism"
)

validation_summary.loc[
    indeterminate_mask, "validation_n"
] = validation_summary.loc[
    indeterminate_mask, "population_n"
]

# Automation target, capped by available population
automation_mask = (
    validation_summary["provisional_label"] == "Automation-oriented"
)

validation_summary.loc[
    automation_mask, "validation_n"
] = validation_summary.loc[
    automation_mask, "population_n"
].clip(upper=AUTOMATION_TARGET)

# Remaining sample goes proportionally to the two majority classes
majority_mask = validation_summary["provisional_label"].isin([
    "Augmentation-oriented",
    "Limited GenAI applicability",
])

remaining_n = (
    VALIDATION_TOTAL
    - validation_summary["validation_n"].sum()
)

majority_population = validation_summary.loc[
    majority_mask, "population_n"
]

majority_allocation = (
    remaining_n
    * majority_population
    / majority_population.sum()
)

# Integer allocation
validation_summary.loc[
    majority_mask, "validation_n"
] = majority_allocation.round().astype(int)

validation_summary["sampling_rate"] = (
    validation_summary["validation_n"]
    / validation_summary["population_n"]
)

display(validation_summary)

print(
    "Total validation sample:",
    validation_summary["validation_n"].sum()
)



Step 5 — Draw the 120-task validation sample

In [ ]:
SEED = 42

sample_sizes = (
    validation_summary
    .set_index("provisional_label")["validation_n"]
    .to_dict()
)
validation_sample = pd.concat(
    [
        validation_frame[
            validation_frame["provisional_label"] == label
        ].sample(
            n=n,
            random_state=SEED
        )
        for label, n in sample_sizes.items()
    ],
    ignore_index=True
)

print(f"Validation sample: {len(validation_sample):,}")
print(f"Unique task IDs: {validation_sample['task_id'].nunique():,}")

display(
    validation_sample["provisional_label"]
    .value_counts()
    .rename("sample_n")
    .to_frame()
)

Step 6 — Record sampling information and freeze the validation sample

In [ ]:
validation_summary["sampling_weight"] = (
    validation_summary["population_n"]
    / validation_summary["validation_n"]
)

validation_sample = validation_sample.merge(
    validation_summary[
        [
            "provisional_label",
            "population_n",
            "validation_n",
            "sampling_rate",
            "sampling_weight",
        ]
    ],
    on="provisional_label",
    how="left",
)

validation_sample["sampling_seed"] = SEED

VALIDATION_SAMPLE_FILE = (
    BASE_DIR
    / "02_Task_Data_and_Classification"
    / "Output"
    "Occupational_Task_Validation_Sample_v6.csv"
)

VALIDATION_SAMPLE_FILE.parent.mkdir(parents=True, exist_ok=True)

if VALIDATION_SAMPLE_FILE.exists():
    print("Validation sample v6 already exists — file not overwritten.")
else:
    validation_sample.to_csv(VALIDATION_SAMPLE_FILE, index=False)
    print(f"Saved and locked: {VALIDATION_SAMPLE_FILE}")
    print(f"Tasks: {len(validation_sample):,}")

Step 7 — Calibrate the two human coders using the excluded pilot tasks

In [ ]:
calibration_tasks = pilot[
    [
        "task_id",
        "soc6",
        "occupation_title",
        "task_statement"
    ]
].copy()

calibration_tasks["coder_label"] = ""
calibration_tasks["coder_notes"] = ""

display(calibration_tasks.head())

In [ ]:
CALIBRATION_DIR = Path(
    "/content/drive/MyDrive/genai-hybrid-skills-adaptation/output/"
    "Occupational/Validation/"
)

calibration_tasks.to_excel(
    BASE_DIR
    / "02_Task_Data_and_Classification"
    / "Output" / "Coder1_Calibration_v6.xlsx",
    index=False
)

calibration_tasks.to_excel(
    BASE_DIR
    / "02_Task_Data_and_Classification"
    / "Output" / "Coder2_Calibration_v6.xlsx",
    index=False
)

print("Coder calibration files saved.")

Step 8 — Create the blind validation files for Coder 1 and Coder 2

In [ ]:
validation_master = pd.read_csv(VALIDATION_SAMPLE_FILE)

blind_validation = validation_master[
    [
        "task_id",
        "soc6",
        "occupation_title",
        "task_statement"
    ]
].copy()

blind_validation["coder_label"] = ""
blind_validation["coder_notes"] = ""

CODER1_FILE = VALIDATION_SAMPLE_FILE.parent / "Coder1_Validation_v6.xlsx"
CODER2_FILE = VALIDATION_SAMPLE_FILE.parent / "Coder2_Validation_v6.xlsx"

blind_validation.to_excel(CODER1_FILE, index=False)
blind_validation.to_excel(CODER2_FILE, index=False)

print(f"Coder 1 file: {CODER1_FILE}")
print(f"Coder 2 file: {CODER2_FILE}")
print(f"Tasks per coder: {len(blind_validation)}")